In [1]:
import numpy as np
import pandas as pd
from sklearn.decomposition import IncrementalPCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import keras
import keras.backend as K
from keras import optimizers
from keras.layers import (Activation, Dense, Dropout)
from keras.layers.normalization import BatchNormalization
from keras.models import Sequential
import argparse
import pickle
import copy 
from scipy import spatial
import tensorflow as tf

Using TensorFlow backend.
/home/siddharth/Siddharth/Compiler/Adversaries-in-IR/pyenv/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:493: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/home/siddharth/Siddharth/Compiler/Adversaries-in-IR/pyenv/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:494: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/home/siddharth/Siddharth/Compiler/Adversaries-in-IR/pyenv/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:495: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.

In [2]:
X = pd.read_csv('./data/testing.csv', sep='\t',header=None)
Y = X.loc[:,0]
X = X.loc[:,1:]
X.columns = range(X.shape[1])

In [3]:
with open('./data/dictionary.pkl', 'rb') as f:
    num_classes = pickle.load(f)
    X_min = pickle.load(f)
    X_max = pickle.load(f)
    ipca=pickle.load(f)
    
x_train = (X - X_min) / (X_max - X_min)
x_train = np.array(x_train)
y_train = np.array(Y)
y_train = y_train - 1
y_train = keras.utils.to_categorical(y_train, num_classes)
x_train = ipca.transform(x_train)

In [4]:
model = keras.models.load_model("./data/last-model.h5")
model.summary()

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense_1 (Dense)              (None, 650)               195650    
_________________________________________________________________
batch_normalization_1 (Batch (None, 650)               2600      
_________________________________________________________________
activation_1 (Activation)    (None, 650)               0         
_________________________________________________________________
dropout_1 (Dropout)          (None, 650)               0         
_________________________________________________________________
dense_2 (Dense)              (None, 600)               390600    
_________________________________________________________________
batch_normalization_2 (Batch (None, 600)               2400      
_________________________________________________________________
activation_2 (Activation)    (None, 600)               0         
__________

In [5]:
score = model.evaluate(x_train, y_train, verbose=0)
print('Test accuracy : {acc:.3f}%'.format(acc=score[1]*100))

Test accuracy : 89.998%


In [6]:
target=model.predict_classes(x_train, batch_size=32, verbose=0)
x_new = []
y_new = []
correct_out = Y -1
for i in range(target.shape[0]):
    if target[i] == correct_out[i]:
        x_new.append(x_train[i])
        y_new.append(target[i])

x_new = np.array(x_new)
y_new = np.array(y_new)
y_new_categorical = keras.utils.to_categorical(y_new, num_classes)
print(x_new.shape, y_new.shape, y_new_categorical.shape)
model.trainable = False

(7450, 300) (7450,) (7450, 104)


In [35]:
learning_rate = .00001
sess = K.get_session()


In [36]:
#value of gradient for the first x_test
for i in range(5):
    print("example #", i)
    x_test_1 = copy.deepcopy(x_new[i:i+1])
    x_orignal = copy.deepcopy(x_new[i:i+1])

    out = (y_new[i+1] + 10) % 104
    out = keras.utils.to_categorical(out, num_classes)
    orignal_y = y_new[i+1]

    for i in range(10):    
        loss = keras.losses.categorical_crossentropy(model.output, tf.convert_to_tensor(out.astype(np.float32)))
        gradients = K.gradients(loss, model.input)
        evaluated_gradients_1 = sess.run(gradients[0], feed_dict={model.input: 
        x_test_1})    
        x_test_1 -= learning_rate*evaluated_gradients_1
        predicted_y = model.predict_classes(x_test_1, batch_size=32, verbose=2)
        print(orignal_y, predicted_y[0], np.argmax(out))
        result = 1 - spatial.distance.cosine(x_test_1, x_orignal)
        print(result)
    #     loss_print =  keras.backend.print_tensor(loss, message='loss')
    #     loss = tf.Print(loss, [loss])
    #     print(loss_print)

example # 0
32 82 42
0.9999999999999869
32 82 42
0.9999999999999903
32 82 42
0.9999999999999932
32 82 42
0.9999999999999953
32 82 42
0.999999999999997
32 82 42
0.9999999999999731
32 82 42
0.9999999999999775
32 82 42
0.9999999999999816
32 82 42
0.9999999999999846
32 82 42
0.9999999999999869
example # 1
60 32 70
0.9999999682636794
60 32 70
0.9999998453660545
60 32 70
0.9999995667037782
60 32 70
0.9999989667787142
60 32 70
0.9999976522027352
60 36 70
0.9999949326025331
60 36 70
0.9999893079283512
60 36 70
0.9999707194181392
60 70 70
0.9993893144915812
60 87 70
0.6219801179165836
example # 2
39 60 49
0.9999999997679438
39 60 49
0.9999999990558763
39 60 49
0.999999997854108
39 60 49
0.9999999957372261
39 60 49
0.9999999926952708
39 60 49
0.9999999885564425
39 60 49
0.9999999827678515
39 60 49
0.9999999753517326
39 60 49
0.9999999655814841
39 60 49
0.9999999520432936
example # 3
97 39 3
0.9999999999999943
97 39 3
0.9999999999999771
97 39 3
0.9999999999999106
97 39 3
0.9999999999998013
97 39 

In [23]:
print(np.argmax(predicted_y))
result = 1 - spatial.distance.cosine(x_test_1, x_orignal)
print(result)

0
0.6219801179165836
